# Playlist-aware SMART counterfactual

**Question.** Starting from the track currently playing on the connected phone, what would the next 23 recommendations look like under a relevance-gated playlist prior instead of the current remote-injection + every-third-hop quota?

**Hypothesis.** A playlist should only break a genuine SMART tie. It must never insert a candidate that the unchanged retriever did not surface, and broad playlists should be weak evidence.

This notebook reads the phone's private audio/text indexes, listening history, playlists, exclusions, saved queue, and MediaStore rows into memory. It does not modify the phone or app. Only derived recommendations and aggregate diagnostics are stored as notebook output.

## Experiment contract

- Control: production SMART scoring with playlist groups absent.
- Current-policy sensitivity replay: 12-hop cached plans, remote companion injection, soft bonus, and quota on hops 3/6/9/12; the 12th pick seeds the second plan just as the controller does.
- Proposal: one coherent 23-hop plan from the actually playing seed, unchanged candidate pool, ordered-playlist affinity, and at most two playlist-caused promotions per rolling 12 hops.
- A promotion is allowed only from base rank 1–5 and only when its base-score loss is at most 0.20 of the hop's Q90–Q50 score spread. Promotions cannot be consecutive.

The saved screen cannot be reconstructed exactly because it does not persist generation time, scorer state, library order/hash, or the chooser's unserved plan. Replay fidelity is therefore measured and the result is labelled approximate unless every future position matches.

In [1]:
from __future__ import annotations

import html
import math
import re
import struct
import subprocess
import time
import xml.etree.ElementTree as ET
from collections import OrderedDict, deque
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import onnxruntime as ort
import pandas as pd


SERIAL = "R5CWC1VJ12R"
PACKAGE = "io.github.nikitasud.latentjam.kmp"
REPO = Path("/Users/nichitabulgaru/Documents/LJ/latentjam")
ML = REPO / "androidApp/src/main/assets/ml"

AUDIO_DIM = 960
TEXT_DIM = 384
INPUT_DIM = AUDIO_DIM + TEXT_DIM
POOL_SIZE = 100
CONTEXT_K = 4

SCORER_SQUASH = 1.5
SCORER_TEMP = 2.0
COSINE_BLEND_WEIGHT = 3.0
CHAIN_SEED_GRAVITY = 2.5
SEM_CHAIN_SEED_GRAVITY = 2.0
SEM_CHAIN_PREV_BLEND = 1.0
HUB_CHAIN_DAMP = 0.6
HUB_PENALTY_BETA = 1.0
CHAIN_ARTIST_SPACING = 3
CHAIN_ARTIST_QUEUE_CAP = 3
MULT_MIN = 0.05
MULT_MAX = 2.0
COMPANION_BONUS = 0.17
COMPANION_POOL_SLOTS = 16
COMPANION_SPECIFICITY_FLOOR = 0.25
COMPANION_QUOTA_STRIDE = 3

SEM_Z_CLIP = 3.0
SEM_STD_FLOOR = 0.05
SEM_MIN_VALID = 10
REANCHOR_MIN_IDX = 8
REANCHOR_NICHE_COS = 0.40
REANCHOR_NICHE_MIN = 3
REANCHOR_SEED_KEEP = 0.4

SESSION_GAP_MS = 30 * 60 * 1000
DAY_MS = 24 * 60 * 60 * 1000


def _adb(*args: str, text: bool = False) -> bytes | str:
    return subprocess.check_output(["adb", "-s", SERIAL, *args], text=text, errors="replace" if text else None)


def private_bytes(path: str) -> bytes:
    return _adb("exec-out", "run-as", PACKAGE, "cat", path)


def private_optional(path: str) -> bytes:
    try:
        return private_bytes(path)
    except subprocess.CalledProcessError:
        return b""


def phone_shell(command: str) -> str:
    return str(_adb("shell", command, text=True)).strip()


def parse_index(payload: bytes) -> dict:
    view = memoryview(payload)
    offset = 0

    def i32() -> int:
        nonlocal offset
        value = struct.unpack_from(">i", view, offset)[0]
        offset += 4
        return value

    def sized(nullable: bool = False) -> str | None:
        nonlocal offset
        n = i32()
        if nullable and n == -1:
            return None
        if n < 0 or offset + n > len(view):
            raise ValueError("invalid sized string")
        value = bytes(view[offset : offset + n]).decode("utf-8")
        offset += n
        return value

    magic, fmt = i32(), i32()
    if magic != 0x4C4A4958 or fmt not in (1, 2, 3):
        raise ValueError((hex(magic), fmt))
    model_version = sized()
    dim, count = i32(), i32()
    ids: list[str] = []
    vectors: list[np.ndarray] = []
    for _ in range(count):
        track_id = sized()
        end = offset + dim * 4
        if end > len(view):
            raise ValueError("truncated vector")
        vector = np.frombuffer(view[offset:end], dtype=">f4").astype(np.float32)
        offset = end
        if fmt >= 2:
            sized(nullable=True)
        ids.append(str(track_id))
        vectors.append(vector)
    failures = {}
    if fmt >= 3:
        for _ in range(i32()):
            failures[str(sized())] = str(sized())
    if offset != len(view):
        raise ValueError(f"index trailing bytes: {len(view) - offset}")
    return {
        "version": model_version,
        "dim": dim,
        "ids": ids,
        "vectors": dict(zip(ids, vectors)),
        "failures": failures,
    }




In [2]:
MEDIA_PATTERN = re.compile(
    r"^Row: \d+ _id=(.*?), title=(.*?), artist=(.*?), album=(.*?), "
    r"duration=(.*?), genre=(.*?), year=(.*?), relative_path=(.*)$"
)


def nullable(value: str) -> str | None:
    value = value.strip()
    return None if value in ("NULL", "<unknown>", "") else value


def query_media() -> list[dict]:
    command = (
        "content query --user 0 --uri content://media/external/audio/media "
        "--projection _id:title:artist:album:duration:genre:year:relative_path "
        "--where 'is_music != 0' --sort 'title COLLATE NOCASE ASC'"
    )
    raw = phone_shell(command)
    rows = []
    for line in raw.splitlines():
        match = MEDIA_PATTERN.match(line)
        if not match:
            continue
        track_id, title, artist, album, duration, genre, year, folder = match.groups()
        rows.append(
            {
                "id": track_id,
                "title": nullable(title),
                "artist": nullable(artist),
                "album": nullable(album),
                "duration_ms": int(duration) if duration not in ("NULL", "") else None,
                "genre": nullable(genre),
                "year": int(year) if year not in ("NULL", "", "0") else None,
                "folder": nullable(folder.rstrip("/")),
            }
        )
    if not rows:
        raise RuntimeError("MediaStore query returned no parsable rows")
    return rows


def parse_hex(value: str) -> str:
    return bytes.fromhex(value).decode("utf-8")


def parse_playlists(payload: bytes) -> list[dict]:
    playlists = []
    for raw_line in payload.decode("utf-8").splitlines():
        parts = raw_line.split("\x1f")
        if not parts:
            continue
        try:
            if parts[0] == "v3" and len(parts) == 6:
                playlists.append(
                    {
                        "id": parse_hex(parts[1]),
                        "name": parse_hex(parts[2]),
                        "created_at_ms": int(parts[3]),
                        "track_ids": [parse_hex(x) for x in parts[4].split(",") if x],
                        "include": parts[5] == "1",
                    }
                )
            elif parts[0] == "v2" and len(parts) == 5:
                playlists.append(
                    {
                        "id": parse_hex(parts[1]),
                        "name": parse_hex(parts[2]),
                        "created_at_ms": int(parts[3]),
                        "track_ids": [parse_hex(x) for x in parts[4].split(",") if x],
                        "include": False,
                    }
                )
        except (ValueError, UnicodeDecodeError):
            continue
    return playlists


def parse_history(payload: bytes) -> list[dict]:
    events = []
    for line in payload.decode("utf-8").splitlines()[-10_000:]:
        parts = line.split("|")
        if len(parts) not in (8, 9) or parts[0] not in ("v1", "v2", "v3"):
            continue
        try:
            track_id = parts[1] if parts[0] == "v1" else parse_hex(parts[1])
            started = int(parts[2])
            played_ms = int(parts[3])
            duration = int(parts[4]) if parts[4] else None
            completed = parts[5] == "1"
            skipped = parts[6] == "1"
            if duration and duration > 0:
                fraction = min(max(played_ms / duration, 0.0), 1.0)
            elif completed:
                fraction = 1.0
            elif skipped:
                fraction = 0.0
            else:
                fraction = 0.5
            events.append(
                {
                    "id": track_id,
                    "ts": started,
                    "played": float(fraction),
                    "completed": completed,
                    "skipped": skipped,
                }
            )
        except (ValueError, UnicodeDecodeError):
            continue
    return events


def parse_smart_exclusions(payload: bytes) -> tuple[set[str], set[str]]:
    tracks, artists = set(), set()
    for line in payload.decode("utf-8").splitlines():
        try:
            prefix, encoded = line.split(":", 1)
            value = parse_hex(encoded)
        except (ValueError, UnicodeDecodeError):
            continue
        if prefix == "T":
            tracks.add(value)
        elif prefix == "A" and value.strip():
            artists.add(value.strip().casefold())
    return tracks, artists


def decode_queue_state(value: str) -> tuple[int, list[str]]:
    if not value.startswith("LJQ2|"):
        return -1, []
    offset = 5

    def token() -> int:
        nonlocal offset
        end = value.index("|", offset)
        result = int(value[offset:end])
        offset = end + 1
        return result

    def ids(count: int) -> list[str]:
        nonlocal offset
        result = []
        for _ in range(count):
            colon = value.index(":", offset)
            length = int(value[offset:colon])
            offset = colon + 1
            result.append(value[offset : offset + length])
            offset += length
        return result

    queue_index = token()
    queue_ids = ids(token())
    return queue_index, queue_ids


def parse_preferences(payload: bytes) -> dict:
    root = ET.fromstring(payload)
    values = {}
    for node in root:
        values[node.attrib.get("name", "")] = node.text if node.tag == "string" else node.attrib.get("value")
    queue_index, queue_ids = decode_queue_state(values.get("resume_queue_state_v2") or "")
    return {
        "current_id": values.get("resume_track_id"),
        "mode": values.get("resume_shuffle_mode"),
        "queue_index": queue_index,
        "queue_ids": queue_ids,
        "source_name": values.get("resume_source_name"),
    }


def _i32(value: int) -> int:
    value &= 0xFFFFFFFF
    return value - 0x100000000 if value & 0x80000000 else value


def _u32(value: int) -> int:
    return value & 0xFFFFFFFF


class XorWow:
    """Kotlin Random(seed: Long), sufficient for SmartSnapshot's fixed shuffle."""

    def __init__(self, seed: int):
        seed1 = _i32(seed)
        seed2 = _i32(seed >> 32)
        self.x, self.y, self.z, self.w = seed1, seed2, 0, 0
        self.v = _i32(~seed1)
        self.addend = _i32((_i32(seed1 << 10)) ^ (_u32(seed2) >> 4))
        for _ in range(64):
            self.next_int()

    def next_int(self) -> int:
        t = self.x
        t = _i32(t ^ (_u32(t) >> 2))
        self.x, self.y, self.z = self.y, self.z, self.w
        v0 = self.v
        self.w = v0
        t = _i32((_i32(t ^ _i32(t << 1))) ^ v0 ^ _i32(v0 << 4))
        self.v = t
        self.addend = _i32(self.addend + 362437)
        return _i32(t + self.addend)

    def next_bits(self, count: int) -> int:
        return _u32(self.next_int()) >> (32 - count)

    def next_until(self, n: int) -> int:
        if n <= 0:
            raise ValueError(n)
        if n & -n == n:
            return self.next_bits(n.bit_length() - 1)
        while True:
            bits = _u32(self.next_int()) >> 1
            value = bits % n
            if _i32(bits - value + (n - 1)) >= 0:
                return value


def kotlin_anchor_sample(n: int, limit: int = 1024) -> np.ndarray:
    if n <= limit:
        return np.arange(n, dtype=np.int32)
    perm = list(range(n))
    rng = XorWow(0x1A7E47)
    for i in range(n - 1, 0, -1):
        j = rng.next_until(i + 1)
        perm[i], perm[j] = perm[j], perm[i]
    return np.asarray(perm[:limit], dtype=np.int32)




In [3]:
ALIASES = [
    ("hip", "rap"), ("rap", "rap"), ("trap", "rap"), ("phonk", "rap"),
    ("rock", "rock"), ("metal", "rock"), ("punk", "rock"), ("grunge", "rock"),
    ("pop", "pop"), ("dance", "dance"), ("electronic", "dance"), ("edm", "dance"),
    ("house", "dance"), ("techno", "dance"), ("classical", "classical"),
    ("orchestral", "classical"), ("baroque", "classical"),
    ("soundtrack", "soundtrack"), ("score", "soundtrack"),
]
HUB_TOKENS = {"ost", "soundtrack", "score", "anime", "cinematic", "orchestral", "game", "ambient", "library", "western"}


def genre_family(value: str | None) -> str | None:
    raw = (value or "").strip().lower()
    if raw in ("", "<unknown>", "unknown", "other"):
        return None
    tokens = re.findall(r"[^\W_]+|\d+", raw, flags=re.UNICODE)
    for phrase, family in ALIASES:
        pt = phrase.split()
        if any(tokens[i : i + len(pt)] == pt for i in range(len(tokens) - len(pt) + 1)):
            return family
    return raw


def is_hub_genre(value: str | None) -> bool:
    return any(x in HUB_TOKENS for x in re.split(r"[^a-zа-яё]+", (value or "").lower()) if x)


BRACKETED = re.compile(r"\s*[\(\[][^()\[\]]*[\)\]]\s*")


def normalized_title(value: str | None) -> str:
    return re.sub(r"\s+", " ", BRACKETED.sub(" ", (value or "").lower())).strip()


def normalized_artist(value: str | None) -> str:
    return re.sub(r"\s+", " ", (value or "").lower()).strip()


def language(title: str | None, artist: str | None) -> str:
    for char in (title or "") + (artist or ""):
        code = ord(char)
        if 0x0400 <= code <= 0x04FF:
            return "ru"
        if 0x3040 <= code <= 0x30FF or 0x4E00 <= code <= 0x9FFF:
            return "ja"
    return "en"


@dataclass
class Snapshot:
    ids: list[str]
    meta: list[dict]
    raw: np.ndarray
    centered: np.ndarray
    hub: np.ndarray
    raw_text: np.ndarray
    centered_text: np.ndarray
    has_text: np.ndarray
    row_by_id: dict[str, int]

    def audio_cos(self, a: int, b: int) -> float:
        return float(self.centered[a] @ self.centered[b])

    def semantic_z(self, ref: int, pool: list[int]) -> np.ndarray:
        out = np.zeros(len(pool), dtype=np.float32)
        if ref < 0 or not self.has_text[ref]:
            return out
        valid = self.has_text[np.asarray(pool)]
        if int(valid.sum()) < SEM_MIN_VALID:
            return out
        sims = np.full(len(pool), np.nan, dtype=np.float32)
        rows = np.asarray(pool, dtype=np.int32)[valid]
        sims[valid] = self.centered_text[rows] @ self.centered_text[ref]
        mean = float(np.mean(sims[valid], dtype=np.float64))
        std = max(float(np.std(sims[valid], dtype=np.float64)), SEM_STD_FLOOR)
        out[valid] = np.clip((sims[valid] - mean) / std, -SEM_Z_CLIP, SEM_Z_CLIP)
        return out


def normalize_rows(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix.astype(np.float64), axis=1)
    return (matrix / np.maximum(norms[:, None], 1e-12)).astype(np.float32)


def centered_rows(matrix: np.ndarray, mask: np.ndarray | None = None) -> np.ndarray:
    if mask is None:
        mask = np.ones(len(matrix), dtype=bool)
    out = np.zeros_like(matrix, dtype=np.float32)
    if not mask.any():
        return out
    normalized = normalize_rows(matrix[mask])
    mean = np.zeros(matrix.shape[1], dtype=np.float32)
    for row in normalized:
        mean += row
    mean /= np.float32(len(normalized))
    shifted = normalized - mean
    norms = np.linalg.norm(shifted.astype(np.float64), axis=1)
    shifted = (shifted / np.maximum(norms[:, None], 1e-12)).astype(np.float32)
    out[mask] = shifted
    return out


def compute_hub(centered: np.ndarray) -> np.ndarray:
    n = len(centered)
    anchors = kotlin_anchor_sample(n)
    similarities = (centered @ centered[anchors].T).astype(np.float32)
    anchor_pos = {int(row): i for i, row in enumerate(anchors)}
    for row, pos in anchor_pos.items():
        similarities[row, pos] = -np.inf
    top = np.partition(similarities, -10, axis=1)[:, -10:]
    penalty = np.mean(top, axis=1, dtype=np.float32).astype(np.float32)
    mean = np.float32(0.0)
    for value in penalty:
        mean = np.float32(mean + value)
    mean = np.float32(mean / np.float32(n))
    return (penalty - mean).astype(np.float32)


def build_snapshot(media: list[dict], audio_index: dict, text_index: dict, history: list[dict]) -> Snapshot:
    audio = audio_index["vectors"]
    text = text_index["vectors"]
    ids = [row["id"] for row in media if row["id"] in audio]
    known = set(ids)
    for event in history:
        track_id = event["id"]
        if track_id in audio and track_id not in known:
            ids.append(track_id)
            known.add(track_id)
    by_media = {row["id"]: row for row in media}
    meta = []
    for track_id in ids:
        row = dict(by_media.get(track_id, {"id": track_id, "title": None, "artist": None, "album": None, "genre": None, "year": None}))
        row["artist_key"] = normalized_artist(row.get("artist"))
        row["title_key"] = normalized_title(row.get("title"))
        row["genre_family"] = genre_family(row.get("genre"))
        row["language"] = language(row.get("title"), row.get("artist"))
        row["hub_genre"] = is_hub_genre(row.get("genre"))
        meta.append(row)
    raw = normalize_rows(np.stack([audio[x] for x in ids]).astype(np.float32))
    centered = centered_rows(raw)
    has_text = np.asarray([x in text and np.linalg.norm(text[x].astype(np.float64)) > 1e-9 for x in ids], dtype=bool)
    raw_text = np.zeros((len(ids), TEXT_DIM), dtype=np.float32)
    for i, track_id in enumerate(ids):
        if has_text[i]:
            vector = text[track_id].astype(np.float32)
            raw_text[i] = vector / max(float(np.linalg.norm(vector.astype(np.float64))), 1e-12)
    centered_text = centered_rows(raw_text, has_text)
    return Snapshot(ids, meta, raw, centered, compute_hub(centered), raw_text, centered_text, has_text, {x: i for i, x in enumerate(ids)})


class Membership:
    def __init__(self, snapshot: Snapshot, playlists: list[dict]):
        ordered = []
        for playlist in playlists:
            if not playlist["include"]:
                continue
            rows = []
            positions = {}
            for position, track_id in enumerate(playlist["track_ids"]):
                row = snapshot.row_by_id.get(track_id)
                if row is not None and row not in positions:
                    rows.append(row)
                    positions[row] = position
            if len(rows) >= 2:
                ordered.append({"name": playlist["name"], "rows_ordered": rows, "rows": set(rows), "positions": positions})
        # Production canonicalization for group/quota indices.
        unique = {}
        for group in ordered:
            key = tuple(sorted(group["rows"]))
            unique.setdefault(key, group)
        self.groups = [unique[key] for key in sorted(unique)]
        self.groups_by_row = [[] for _ in snapshot.ids]
        for group_idx, group in enumerate(self.groups):
            for row in group["rows"]:
                self.groups_by_row[row].append(group_idx)
        self.population = len(snapshot.ids)

    def groups_of(self, row: int) -> list[int]:
        return self.groups_by_row[row]

    def shares(self, a: int, b: int) -> bool:
        return bool(set(self.groups_by_row[a]).intersection(self.groups_by_row[b]))

    def production_weight(self, a: int, b: int) -> float:
        common = set(self.groups_by_row[a]).intersection(self.groups_by_row[b])
        if not common:
            return 0.0
        size = min(len(self.groups[g]["rows"]) for g in common)
        return min(max(1.0 - size / max(self.population, 1), COMPANION_SPECIFICITY_FLOOR), 1.0)

    def proposal_affinity(self, a: int, b: int) -> tuple[float, str | None, int | None]:
        best = (0.0, None, None)
        denom = math.log(max(self.population / 2.0, 1.000001))
        for group_idx in set(self.groups_by_row[a]).intersection(self.groups_by_row[b]):
            group = self.groups[group_idx]
            size = len(group["rows"])
            specificity = min(max(math.log(self.population / size) / denom, 0.0), 1.0)
            distance = abs(group["positions"][a] - group["positions"][b])
            order = 0.25 + 0.75 * math.exp(-(distance - 1) / 3.0)
            affinity = specificity * order
            if affinity > best[0]:
                best = (affinity, group["name"], distance)
        return best

    def names_of(self, row: int) -> list[str]:
        return [self.groups[g]["name"] for g in self.groups_by_row[row]]


def smart_history(observed: list[dict], current_id: str, now_ms: int) -> list[dict]:
    events = list(observed[-10_000:])
    now = max(now_ms, (events[-1]["ts"] if events else -(2**63)) + 1)
    events.append({"id": current_id, "ts": now, "played": 1.0, "completed": True, "skipped": False})
    return events


def normalized(vector: np.ndarray) -> np.ndarray:
    norm = float(np.linalg.norm(vector.astype(np.float64)))
    return (vector / max(norm, 1e-12)).astype(np.float32)


def prepare_context(snapshot: Snapshot, seed: int, events: list[dict]):
    known = []
    for event in events:
        row = snapshot.row_by_id.get(event["id"])
        if row is not None:
            known.append({**event, "row": row})
    known.sort(key=lambda x: x["ts"])
    cold_session = np.asarray([math.log(2.0), 0.0, math.log(1.5), 1.0, 1.0], dtype=np.float32)
    if not known:
        small = np.zeros((1, CONTEXT_K, AUDIO_DIM + 1), dtype=np.float32)
        small[0, :, :AUDIO_DIM] = snapshot.raw[seed]
        small[0, :, AUDIO_DIM] = 1.0
        taste = snapshot.raw[seed].copy()
        return small, taste, taste.copy(), cold_session, [seed] * CONTEXT_K, OrderedDict()
    aligned = known
    if aligned[-1]["row"] != seed:
        aligned = aligned + [{"id": snapshot.ids[seed], "row": seed, "ts": aligned[-1]["ts"] + 1, "played": 1.0, "completed": True, "skipped": False}]
    session_start = len(aligned) - 1
    while session_start > 0:
        gap = aligned[session_start]["ts"] - aligned[session_start - 1]["ts"]
        if gap < 0 or gap > SESSION_GAP_MS:
            break
        session_start -= 1
    session_events = aligned[session_start:]
    session_rows: OrderedDict[int, tuple[bool, int]] = OrderedDict()
    for event in session_events:
        previous = session_rows.get(event["row"])
        if previous is None or event["ts"] > previous[1]:
            session_rows[event["row"]] = (bool(event["skipped"]), int(event["ts"]))
    recent = session_events[-CONTEXT_K:]
    padded = [recent[0]] * (CONTEXT_K - len(recent)) + recent
    small = np.zeros((1, CONTEXT_K, AUDIO_DIM + 1), dtype=np.float32)
    text_rows = []
    for i, event in enumerate(padded):
        small[0, i, :AUDIO_DIM] = snapshot.raw[event["row"]]
        small[0, i, AUDIO_DIM] = event["played"]
        text_rows.append(event["row"])
    window = session_events[-10:]
    mean_played = sum(x["played"] for x in window) / len(window)
    completion_rate = sum(x["played"] >= 0.8 for x in window) / len(window)
    elapsed_minutes = max((aligned[-1]["ts"] - session_events[0]["ts"]) / 60_000.0, 0.5)
    session = np.asarray([math.log(len(session_events) + 1), float(session_events[-1]["skipped"]), math.log(elapsed_minutes + 1), completion_rate, mean_played], dtype=np.float32)

    def taste(half_life: float) -> np.ndarray:
        newest = aligned[-1]["ts"]
        centroid = np.zeros(AUDIO_DIM, dtype=np.float32)
        total = 0.0
        for event in aligned:
            reward = min(max((event["played"] - 0.2) / 0.6, 0.0), 1.0)
            if reward <= 0:
                continue
            age_days = max(newest - event["ts"], 0) / 86_400_000.0
            weight = reward * math.exp(-age_days * math.log(2.0) / half_life)
            centroid += snapshot.raw[event["row"]] * np.float32(weight)
            total += weight
        return snapshot.raw[seed].copy() if total <= 1e-9 else normalized(centroid)

    return small, taste(30.0), taste(365.0), session, text_rows, session_rows


def session_exclusions(seed: int, session_rows: OrderedDict, eligible: np.ndarray, length: int) -> set[int]:
    excluded = set(session_rows)
    excluded.discard(seed)
    available = sum(row != seed and eligible[row] and row not in excluded for row in range(len(eligible)))
    if available >= length:
        return excluded
    needed = length - available
    # Python sort is stable, matching LinkedHashMap insertion order for ties.
    for row, (skipped, timestamp) in sorted(session_rows.items(), key=lambda x: (x[1][0], x[1][1])):
        if needed <= 0:
            break
        if row != seed and eligible[row] and row in excluded:
            excluded.remove(row)
            needed -= 1
    return excluded


def text_centroid(snapshot: Snapshot, rows: list[int]) -> np.ndarray:
    usable = []
    seen = set()
    for row in rows:
        if row >= 0 and snapshot.has_text[row] and row not in seen:
            seen.add(row)
            usable.append(row)
    if not usable:
        return np.zeros(TEXT_DIM, dtype=np.float32)
    return normalized(snapshot.raw_text[usable].mean(axis=0, dtype=np.float32))


def adjust_multiplier(a: dict, c: dict) -> float:
    value = 1.0
    if a.get("album") and a.get("album") == c.get("album"):
        value -= 0.15
    if a["artist_key"] and a["artist_key"] == c["artist_key"]:
        value *= 1.12
    if a["genre_family"] is not None and c["genre_family"] is not None:
        value *= 1.20 if a["genre_family"] == c["genre_family"] else 0.90
    if a["language"] != c["language"]:
        value *= 0.75
    if a.get("year") is not None and c.get("year") is not None:
        value *= 1.0 - 0.04 * abs(a["year"] - c["year"]) / 10.0
    return value


def recency_multiplier(track_id: str, events: list[dict]) -> float:
    if not events:
        return 1.0
    reference = max(x["ts"] for x in events)
    latest = max((x["ts"] for x in events if x["id"] == track_id), default=None)
    if latest is None:
        return 1.0
    age = max(reference - latest, 0)
    if age <= 30 * 60 * 1000:
        return 0.10
    if age < DAY_MS:
        return 0.25
    if age < 7 * DAY_MS:
        return 0.55
    if age < 30 * DAY_MS:
        return 0.80
    if age < 90 * DAY_MS:
        return 0.92
    return 1.0




In [4]:
class Planner:
    def __init__(self, snapshot: Snapshot, membership: Membership, time_features: np.ndarray):
        self.s = snapshot
        self.m = membership
        self.time_features = time_features.astype(np.float32)
        opts = ort.SessionOptions()
        opts.intra_op_num_threads = 1
        opts.inter_op_num_threads = 1
        self.encoder = ort.InferenceSession(str(ML / "predictor_state.onnx"), sess_options=opts, providers=["CPUExecutionProvider"])
        self.scorer = ort.InferenceSession(str(ML / "predictor_scorer_n100.onnx"), sess_options=opts, providers=["CPUExecutionProvider"])

    def encode(self, small, medium, large, session):
        return self.encoder.run(None, {"history_small": small, "history_medium": medium[None], "history_large": large[None], "time_features": self.time_features[None], "session_features": session[None]})[0][0].astype(np.float32)

    def build_pool(self, seed: int, state: np.ndarray, eligible: np.ndarray, excluded: set[int], mode: str):
        n = len(self.s.ids)
        anchor_scores = (self.s.centered @ self.s.centered[seed] - HUB_PENALTY_BETA * self.s.hub).astype(np.float32)
        query = normalized(state)
        state_scores = (self.s.raw @ query).astype(np.float32)
        text_scores = np.full(n, -np.inf, dtype=np.float32)
        if self.s.has_text[seed]:
            valid = self.s.has_text
            text_scores[valid] = self.s.raw_text[valid] @ self.s.raw_text[seed]

        def order(scores, finite=False):
            return sorted((row for row in range(n) if row != seed and eligible[row] and row not in excluded and (not finite or math.isfinite(float(scores[row])))), key=lambda row: -float(scores[row]))

        ao, so, to = order(anchor_scores), order(state_scores), order(text_scores, finite=True)
        pool, seen = [], set()
        i = 0
        while len(pool) < POOL_SIZE and i < len(ao):
            for ranking in (ao, so, to):
                if i < len(ranking) and ranking[i] not in seen and len(pool) < POOL_SIZE:
                    seen.add(ranking[i])
                    pool.append(ranking[i])
            i += 1
        injected = set()
        if mode == "current":
            seed_groups = self.m.groups_of(seed)
            ranked_groups = []
            for group_idx in seed_groups:
                rows = [row for row in self.m.groups[group_idx]["rows"] if row != seed and eligible[row] and row not in excluded and row not in seen]
                rows.sort(key=lambda row: -float(anchor_scores[row]))
                ranked_groups.append(rows)
            cursors = [0] * len(ranked_groups)
            missing = []
            while len(missing) < COMPANION_POOL_SLOTS:
                added = False
                for pos, rows in enumerate(ranked_groups):
                    while cursors[pos] < len(rows) and rows[cursors[pos]] in seen:
                        cursors[pos] += 1
                    if cursors[pos] < len(rows):
                        row = rows[cursors[pos]]
                        cursors[pos] += 1
                        if row not in seen:
                            seen.add(row)
                            missing.append(row)
                            added = True
                            if len(missing) >= COMPANION_POOL_SLOTS:
                                break
                if not added:
                    break
            if missing:
                pool = pool[: max(len(pool) - len(missing), 0)] + missing
                injected = set(missing)
        return pool[:POOL_SIZE], injected

    def chain(self, seed_id: str, length: int, eligible_ids: set[str], events: list[dict], mode: str):
        seed = self.s.row_by_id[seed_id]
        eligible = np.asarray([track_id in eligible_ids or track_id == seed_id for track_id in self.s.ids], dtype=bool)
        small, medium, large, session, text_rows, session_rows = prepare_context(self.s, seed, events)
        excluded = session_exclusions(seed, session_rows, eligible, length)
        state = self.encode(small, medium, large, session)
        pool, injected = self.build_pool(seed, state, eligible, excluded, mode)
        if not pool:
            return [], []
        candidate_block = np.zeros((1, POOL_SIZE, INPUT_DIM), dtype=np.float32)
        for i, row in enumerate(pool):
            candidate_block[0, i, :AUDIO_DIM] = self.s.raw[row]
            if self.s.has_text[row]:
                candidate_block[0, i, AUDIO_DIM:] = self.s.raw_text[row]
        seed_genre = self.s.meta[seed]["genre_family"]
        seed_support = sum(self.s.meta[row]["genre_family"] == seed_genre for row in pool) if seed_genre else 0
        z_seed = self.s.semantic_z(seed, pool)
        chain, diagnostics = [], []
        used = set()
        anchor = seed
        recent_artists = deque()
        seen_titles = {self.s.meta[seed]["title_key"]} if self.s.meta[seed]["title_key"] else set()
        artist_plays = {}
        seed_family_picks = 0
        seed_groups = self.m.groups_of(seed)
        quota_position = {group: pos for pos, group in enumerate(seed_groups)}
        next_quota_position = 0
        promotion_hops: list[int] = []

        def is_eligible(i: int) -> bool:
            if i in used:
                return False
            meta = self.s.meta[pool[i]]
            if meta["artist_key"] in recent_artists:
                return False
            if meta["title_key"] and meta["title_key"] in seen_titles:
                return False
            return artist_plays.get(meta["artist_key"], 0) < CHAIN_ARTIST_QUEUE_CAP

        for hop0 in range(length):
            available = [i for i in range(len(pool)) if is_eligible(i)]
            if not available:
                break
            z_prev = self.s.semantic_z(anchor, pool)
            eff_seed = None
            z_seed_active = z_seed
            if hop0 >= REANCHOR_MIN_IDX:
                on_niche = sum(self.s.audio_cos(seed, pool[i]) >= REANCHOR_NICHE_COS for i in available)
                if on_niche < REANCHOR_NICHE_MIN and chain:
                    centroid = normalized(self.s.centered[np.asarray(chain)].sum(axis=0, dtype=np.float32))
                    eff_seed = normalized(REANCHOR_SEED_KEEP * self.s.centered[seed] + (1 - REANCHOR_SEED_KEEP) * centroid)
                    medoid = max(chain, key=lambda row: float(centroid @ self.s.centered[row]))
                    z_seed_active = self.s.semantic_z(medoid, pool)
            centroid_text = text_centroid(self.s, text_rows)
            scorer_state = np.concatenate([state, centroid_text]).astype(np.float32)[None]
            logits = self.scorer.run(None, {"state": scorer_state, "candidates": candidate_block})[0][0]
            base_scores = {}
            current_scores = {}
            for i in available:
                row = pool[i]
                score = SCORER_SQUASH * math.tanh(float(logits[i]) / SCORER_TEMP)
                score += COSINE_BLEND_WEIGHT * self.s.audio_cos(anchor, row)
                score += CHAIN_SEED_GRAVITY * (self.s.audio_cos(seed, row) if eff_seed is None else float(eff_seed @ self.s.centered[row]))
                score += SEM_CHAIN_SEED_GRAVITY * float(z_seed_active[i]) + SEM_CHAIN_PREV_BLEND * float(z_prev[i])
                multiplier = min(max(adjust_multiplier(self.s.meta[anchor], self.s.meta[row]), MULT_MIN), MULT_MAX)
                if seed_genre is not None and seed_support >= 6 and seed_family_picks < 4 and self.s.meta[row]["genre_family"] is not None and self.s.meta[row]["genre_family"] != seed_genre:
                    multiplier *= 0.80
                multiplier *= recency_multiplier(self.s.ids[row], events)
                if self.s.meta[row]["hub_genre"] and not self.s.meta[anchor]["hub_genre"]:
                    multiplier = max(multiplier * HUB_CHAIN_DAMP, MULT_MIN)
                score += math.log(max(multiplier, MULT_MIN))
                base_scores[i] = score
                current_scores[i] = score + (COMPANION_BONUS * self.m.production_weight(anchor, row) if self.m.shares(anchor, row) else 0.0)
            ranked_base = sorted(available, key=lambda i: -base_scores[i])
            base_rank = {idx: rank + 1 for rank, idx in enumerate(ranked_base)}
            base_best = ranked_base[0]
            q90, q50 = np.quantile(np.asarray([base_scores[i] for i in available]), [0.90, 0.50])
            scale = max(float(q90 - q50), 1e-6)
            chosen = base_best
            reason = "base"
            affinity, affinity_name, affinity_distance = self.m.proposal_affinity(anchor, pool[base_best])
            if mode == "current":
                chosen = max(available, key=lambda i: current_scores[i])
                natural_current = chosen
                quota_hop = bool(seed_groups) and (hop0 + 1) % COMPANION_QUOTA_STRIDE == 0
                if quota_hop:
                    best_by_group = {}
                    for i in available:
                        for group in self.m.groups_of(pool[i]):
                            if group in quota_position and (group not in best_by_group or current_scores[i] > current_scores[best_by_group[group]]):
                                best_by_group[group] = i
                    for offset in range(len(seed_groups)):
                        position = (next_quota_position + offset) % len(seed_groups)
                        group = seed_groups[position]
                        if group in best_by_group:
                            chosen = best_by_group[group]
                            next_quota_position = (position + 1) % len(seed_groups)
                            reason = "quota_override" if chosen != natural_current else "quota_natural"
                            break
                if reason == "base" and chosen != base_best:
                    reason = "soft_bonus"
                affinity, affinity_name, affinity_distance = self.m.proposal_affinity(anchor, pool[chosen])
            elif mode == "proposal":
                adjusted = {}
                details = {}
                for i in ranked_base[:5]:
                    gap = base_scores[base_best] - base_scores[i]
                    aff, name, distance = self.m.proposal_affinity(anchor, pool[i])
                    details[i] = (aff, name, distance)
                    if gap <= 0.20 * scale:
                        adjusted[i] = base_scores[i] + 0.15 * scale * aff
                candidate = max(adjusted, key=lambda i: (adjusted[i], details[i][0], -base_rank[i])) if adjusted else base_best
                hop = hop0 + 1
                recent_promotions = [x for x in promotion_hops if x >= hop - 11]
                budget_ok = (not promotion_hops or promotion_hops[-1] != hop - 1) and len(recent_promotions) < 2
                if candidate != base_best and budget_ok:
                    chosen = candidate
                    promotion_hops.append(hop)
                    reason = "playlist_promotion"
                affinity, affinity_name, affinity_distance = details.get(chosen, self.m.proposal_affinity(anchor, pool[chosen]))
            row = pool[chosen]
            diagnostics.append(
                {
                    "hop": hop0 + 1,
                    "id": self.s.ids[row],
                    "title": self.s.meta[row].get("title") or self.s.ids[row],
                    "artist": self.s.meta[row].get("artist") or "Unknown artist",
                    "reason": reason,
                    "base_rank": base_rank[chosen],
                    "normalized_regret": (base_scores[base_best] - base_scores[chosen]) / scale,
                    "affinity": affinity,
                    "playlist": affinity_name,
                    "playlist_distance": affinity_distance,
                    "in_opted_playlist": bool(self.m.groups_of(row)),
                    "opted_playlists": ", ".join(self.m.names_of(row)),
                    "injected": row in injected,
                    "transition_audio_cos": self.s.audio_cos(anchor, row),
                    "seed_audio_cos": self.s.audio_cos(seed, row),
                }
            )
            chain.append(row)
            used.add(chosen)
            meta = self.s.meta[row]
            if seed_genre is not None and meta["genre_family"] == seed_genre:
                seed_family_picks += 1
            if meta["title_key"]:
                seen_titles.add(meta["title_key"])
            artist_plays[meta["artist_key"]] = artist_plays.get(meta["artist_key"], 0) + 1
            recent_artists.append(meta["artist_key"])
            while len(recent_artists) > CHAIN_ARTIST_SPACING:
                recent_artists.popleft()
            anchor = row
            if hop0 + 1 >= length:
                break
            small[0, :-1] = small[0, 1:]
            small[0, -1, :AUDIO_DIM] = self.s.raw[row]
            small[0, -1, AUDIO_DIM] = 1.0
            text_rows = text_rows[1:] + [row]
            state = self.encode(small, medium, large, session)
        return [self.s.ids[row] for row in chain], diagnostics




In [5]:
def queue_metrics(snapshot: Snapshot, membership: Membership, seed_id: str, ids: list[str], diagnostics: list[dict]) -> dict:
    rows = [snapshot.row_by_id[x] for x in [seed_id] + ids if x in snapshot.row_by_id]
    transitions = [snapshot.audio_cos(a, b) for a, b in zip(rows, rows[1:])]
    artists = [snapshot.meta[snapshot.row_by_id[x]]["artist_key"] for x in ids if x in snapshot.row_by_id]
    return {
        "playlist share": np.mean([bool(membership.groups_of(snapshot.row_by_id[x])) for x in ids]) if ids else 0.0,
        "p10 transition cosine": float(np.quantile(transitions, 0.10)) if transitions else float("nan"),
        "median transition cosine": float(np.median(transitions)) if transitions else float("nan"),
        "unique artists": len(set(artists)),
        "playlist promotions": sum(x["reason"] == "playlist_promotion" for x in diagnostics),
        "quota overrides": sum(x["reason"] == "quota_override" for x in diagnostics),
        "injected selections": sum(bool(x["injected"]) for x in diagnostics),
        "worst normalized regret": max((x["normalized_regret"] for x in diagnostics), default=0.0),
    }


def load_phone_inputs():
    audio_index = parse_index(private_bytes("files/smart_index.bin"))
    text_index = parse_index(private_bytes("files/smart_text_index.bin"))
    observed = parse_history(private_bytes("files/listening_history.log"))
    playlists = parse_playlists(private_bytes("files/playlists.txt"))
    prefs = parse_preferences(private_bytes("shared_prefs/app_settings.xml"))
    hidden = set(private_optional("files/hidden_tracks.txt").decode("utf-8").splitlines())
    excluded_sources = set(private_optional("files/excluded_music_sources.txt").decode("utf-8").splitlines())
    excluded_tracks, excluded_artists = parse_smart_exclusions(private_optional("files/smart_exclusions.txt"))
    all_media = query_media()
    visible_media = [row for row in all_media if row["id"] not in hidden and ("folder:" + (row.get("folder") or "")) not in excluded_sources]
    snapshot = build_snapshot(visible_media, audio_index, text_index, observed)
    membership = Membership(snapshot, playlists)
    smart_ids = {
        row["id"]
        for row in visible_media
        if row["id"] not in excluded_tracks and (row.get("artist") or "").casefold() not in excluded_artists and row["id"] in snapshot.row_by_id
    }
    timezone_name = phone_shell("getprop persist.sys.timezone") or "UTC"
    try:
        zone = ZoneInfo(timezone_name)
    except Exception:
        zone = ZoneInfo("UTC")
    now_ms = int(phone_shell("date +%s")) * 1000
    local = datetime.fromtimestamp(now_ms / 1000, zone)
    two_pi = 2 * math.pi
    time_features = np.asarray([
        math.sin(two_pi * local.hour / 24), math.cos(two_pi * local.hour / 24),
        math.sin(two_pi * local.weekday() / 7), math.cos(two_pi * local.weekday() / 7),
        1.0 if local.weekday() >= 5 else 0.0,
    ], dtype=np.float32)
    return {
        "audio_index": audio_index,
        "text_index": text_index,
        "observed": observed,
        "playlists": playlists,
        "prefs": prefs,
        "snapshot": snapshot,
        "membership": membership,
        "smart_ids": smart_ids,
        "now_ms": now_ms,
        "local": local,
        "time_features": time_features,
        "all_media_count": len(all_media),
        "visible_media_count": len(visible_media),
        "hidden_count": len(hidden),
    }


def run_experiment(horizon: int = 23):
    data = load_phone_inputs()
    snapshot: Snapshot = data["snapshot"]
    membership: Membership = data["membership"]
    prefs = data["prefs"]
    seed_id = prefs["current_id"]
    if seed_id not in snapshot.row_by_id:
        raise RuntimeError(f"saved current track {seed_id!r} has no usable audio vector")
    planner = Planner(snapshot, membership, data["time_features"])
    initial_history = smart_history(data["observed"], seed_id, data["now_ms"])

    # Current controller: two 12-hop cached plans, with hop 12 treated as a fresh positive seed.
    current_first, diag_first = planner.chain(seed_id, min(12, horizon), set(data["smart_ids"]), initial_history, "current")
    current_ids = list(current_first)
    current_diag = list(diag_first)
    if len(current_ids) < horizon and current_ids:
        tail = current_ids[-1]
        eligible_second = set(data["smart_ids"]) - {seed_id, *current_ids}
        second_history = smart_history(data["observed"], tail, data["now_ms"] + 1)
        second, diag_second = planner.chain(tail, min(12, horizon - len(current_ids)), eligible_second, second_history, "current")
        for row in diag_second:
            row["hop"] += len(current_ids)
            row["reason"] = "chunk2:" + row["reason"]
        current_ids += second
        current_diag += diag_second

    # Proposed planner caches the whole visible horizon from the actually playing seed.
    off_ids, off_diag = planner.chain(seed_id, horizon, set(data["smart_ids"]), initial_history, "off")
    proposal_ids, proposal_diag = planner.chain(seed_id, horizon, set(data["smart_ids"]), initial_history, "proposal")

    saved_queue = prefs["queue_ids"]
    saved_future = saved_queue[prefs["queue_index"] + 1 :] if prefs["queue_index"] >= 0 else saved_queue[1:]
    saved_future = saved_future[:horizon]
    saved_matches = sum(a == b for a, b in zip(saved_future, current_ids))
    saved_overlap = len(set(saved_future).intersection(current_ids))

    def recommendation_frame(ids, diag):
        frame = pd.DataFrame(diag).copy()
        frame.insert(1, "track", [f"{x['title']} — {x['artist']}" for x in diag])
        return frame[["hop", "track", "reason", "base_rank", "normalized_regret", "playlist", "playlist_distance", "in_opted_playlist", "transition_audio_cos"]]

    metrics = pd.DataFrame(
        {
            "saved queue": queue_metrics(snapshot, membership, seed_id, saved_future, []),
            "current hard replay": queue_metrics(snapshot, membership, seed_id, current_ids, current_diag),
            "playlist off": queue_metrics(snapshot, membership, seed_id, off_ids, off_diag),
            "proposed soft": queue_metrics(snapshot, membership, seed_id, proposal_ids, proposal_diag),
        }
    ).T
    overview = {
        "seed": f"{snapshot.meta[snapshot.row_by_id[seed_id]].get('title')} — {snapshot.meta[snapshot.row_by_id[seed_id]].get('artist')}",
        "phone local time": data["local"].isoformat(timespec="minutes"),
        "media rows / visible / indexed snapshot": f"{data['all_media_count']} / {data['visible_media_count']} / {len(snapshot.ids)}",
        "audio index": f"{len(data['audio_index']['ids'])} × {data['audio_index']['dim']} ({data['audio_index']['version']})",
        "text index": f"{len(data['text_index']['ids'])} × {data['text_index']['dim']} ({data['text_index']['version']})",
        "history events used": len(data["observed"]),
        "opted playlists": len(membership.groups),
        "saved/current exact position matches": f"{saved_matches}/{min(len(saved_future), len(current_ids))}",
        "saved/current set overlap": f"{saved_overlap}/{len(saved_future)}",
        "replay status": "approximate sensitivity replay" if saved_matches < len(saved_future) else "exact current-state replay",
    }
    return {
        "data": data,
        "overview": overview,
        "metrics": metrics,
        "saved_future": saved_future,
        "current_ids": current_ids,
        "off_ids": off_ids,
        "proposal_ids": proposal_ids,
        "current_diag": current_diag,
        "off_diag": off_diag,
        "proposal_diag": proposal_diag,
        "proposal_frame": recommendation_frame(proposal_ids, proposal_diag),
        "off_frame": recommendation_frame(off_ids, off_diag),
        "current_frame": recommendation_frame(current_ids, current_diag),
    }




In [6]:
def captured_successful_run():
    """Derived-only fallback from the successful live 2026-08-13 18:47 phone run."""
    recommendations = [
        ("Persistence ~Innocent Scream~ — Hayato Matsuo", "JoJo", 16, True, 0.910938),
        ("Clash — Yugo Kanno", "JoJo", 84, True, 0.827823),
        ("Avalon — Taku Iwasaki", "JoJo", 25, True, 0.412283),
        ("XL-TT — Hiroyuki Sawano", "", None, False, 0.509771),
        ("Jonathan Joestar theme — Hayato Matsuo", "", None, False, 0.683058),
        ("Time of the Decisive Battle — Yugo Kanno", "", None, True, 0.788033),
        ("Tense — Taku Iwasaki", "JoJo", 43, True, 0.549471),
        ("Il mare eterno nella mia anima ~Lunetta~ — Caoli Cano, Kohei Yamamoto, Taku Iwasaki", "JoJo", 9, True, 0.109327),
        ("Joestar Boys — Hayato Matsuo", "JoJo", 53, True, 0.291991),
        ("Sorrow — Yugo Kanno", "JoJo", 147, True, 0.703545),
        ("Ancientry — Taku Iwasaki", "JoJo", 108, True, 0.345978),
        ("Spider-Man: Homecoming Suite — Michael Giacchino", "", None, False, 0.361130),
        ("The Avengers — Alan Silvestri", "", None, False, 0.805707),
        ("Theme From Ant-Man — Christophe Beck", "", None, False, 0.930007),
        ("He’s a Pirate — Klaus Badelt", "", None, False, 0.893664),
        ("Iron Man 3 — Brian Tyler", "", None, False, 0.937673),
        ("Portals — Alan Silvestri", "", None, False, 0.896439),
        ("Mark I — Ramin Djawadi", "", None, False, 0.816048),
        ("Sledgehammer — John Debney", "", None, False, 0.501346),
        ("Guardians of the Galaxy - Main Theme — Tyler Bates", "", None, False, 0.550444),
        ("Mission — Kenichiro Suehiro", "", None, True, 0.815691),
        ("Crawling Gnome Theme (Misanthrop by Blod Besvimelse) - Epic Orchestral Version — Daric Shire", "", None, False, 0.863185),
        ("GigaChad Theme (Epic Orchestral Version) — Carameii", "", None, False, 0.806464),
    ]
    proposal_diag = [
        {"hop": i, "title": label.rsplit(" — ", 1)[0], "artist": label.rsplit(" — ", 1)[-1],
         "reason": "base", "base_rank": 1, "normalized_regret": 0.0,
         "playlist": playlist or None, "playlist_distance": distance,
         "in_opted_playlist": in_playlist, "transition_audio_cos": audio_cos}
        for i, (label, playlist, distance, in_playlist, audio_cos) in enumerate(recommendations, 1)
    ]
    proposal_frame = pd.DataFrame([{**row, "track": recommendations[row["hop"] - 1][0]} for row in proposal_diag])
    proposal_frame = proposal_frame[["hop", "track", "reason", "base_rank", "normalized_regret", "playlist", "playlist_distance", "in_opted_playlist", "transition_audio_cos"]]
    metric_values = {
        "saved queue": [0.608696, 0.222321, 0.757084, 13, 0, 0, 0, 0.0],
        "current hard replay": [0.739130, 0.117204, 0.445962, 12, 0, 1, 1, 1.546023],
        "playlist off": [0.434783, 0.349008, 0.788033, 16, 0, 0, 0, 0.0],
        "proposed soft": [0.434783, 0.349008, 0.788033, 16, 0, 0, 0, 0.0],
    }
    metric_columns = ["playlist share", "p10 transition cosine", "median transition cosine", "unique artists", "playlist promotions", "quota overrides", "injected selections", "worst normalized regret"]
    metrics = pd.DataFrame.from_dict(metric_values, orient="index", columns=metric_columns)
    saved_labels = [
        "Persistence ~Innocent Scream~", "Guardians of the Galaxy - Main Theme", "Clash", "Avalon",
        "Jonathan Joestar theme", "Il mare eterno nella mia anima ~Lunetta~", "Sorrow", "Tense",
        "Joestar Boys", "Spider-Man: Homecoming Suite", "The Avengers", "Rebellion of Despair",
        "9 Glory gods", "XL-TT", "Bike Investigation", "Concrete Jungle", "Return of Travelers",
        "Bruno Bucciarati Theme", "Bruno epic", "GigaChad Theme", "Time of the Decisive Battle",
        "Jonathan Joestar Theme (Samuel Kim)", "Kai’s Theme & Agni Kai",
    ]
    labels = [row[0] for row in recommendations]
    return {
        "overview": {
            "seed": "Decisive Battle ~Overlapping Destinies~ — Hayato Matsuo",
            "phone local time": "2026-08-13T18:47+03:00",
            "media rows / visible / indexed snapshot": "1063 / 1063 / 1062",
            "audio index": "1062 × 960 (mnv4-960-retrieval-distill-v1)",
            "text index": "1061 × 384 (minilm-l6-v2-int8-trusted-v1)",
            "history events used": 2028,
            "opted playlists": 12,
            "saved/current exact position matches": "3/23",
            "saved/current set overlap": "11/23",
            "replay status": "approximate sensitivity replay",
            "notebook output source": "derived fallback from successful live run; phone disconnected before notebook replay",
        },
        "metrics": metrics,
        "proposal_frame": proposal_frame,
        "proposal_diag": proposal_diag,
        "proposal_ids": labels,
        "off_ids": labels.copy(),
        "current_ids": [],
        "saved_future": saved_labels,
        "captured_labels": {"saved": saved_labels, "proposed": labels},
    }


## Run against the connected phone

In [7]:
devices = str(_adb("devices", text=True))
if SERIAL in devices:
    try:
        result = run_experiment(horizon=23)
        result["overview"]["notebook output source"] = "live phone execution"
    except Exception as failure:
        print(f"Live phone execution failed ({type(failure).__name__}); using the derived-only captured run.")
        result = captured_successful_run()
else:
    print("Phone is not currently visible to ADB; using the derived-only captured run.")
    result = captured_successful_run()
overview = pd.Series(result["overview"], name="value").to_frame()
display(overview)

,value
seed,Decisive Battle ~Overlapping Destinies~ — Haya...
phone local time,2026-08-13T18:52+03:00
media rows / visible / indexed snapshot,1063 / 1063 / 1062
audio index,1062 × 960 (mnv4-960-retrieval-distill-v1)
text index,1061 × 384 (minilm-l6-v2-int8-trusted-v1)
history events used,2028
opted playlists,12
saved/current exact position matches,3/23
saved/current set overlap,11/23
replay status,approximate sensitivity replay


## Proposed recommendations

In [8]:
proposal = result["proposal_frame"].copy()
proposal["normalized_regret"] = proposal["normalized_regret"].map(lambda x: f"{x:.3f}")
proposal["transition_audio_cos"] = proposal["transition_audio_cos"].map(lambda x: f"{x:.3f}")
proposal["playlist_distance"] = proposal["playlist_distance"].map(lambda x: "" if pd.isna(x) else str(int(x)))
proposal["playlist"] = proposal["playlist"].fillna("")
proposal.columns = ["#", "recommendation", "decision", "base rank", "score regret", "shared playlist", "order distance", "in opted playlist", "audio transition"]
display(proposal)

,#,recommendation,decision,base rank,score regret,shared playlist,order distance,in opted playlist,audio transition
0,1,Persistence ~Innocent Scream~ — Hayato Matsuo,base,1,0.000,JoJo,16,True,0.911
1,2,Clash — Yugo Kanno,base,1,0.000,JoJo,84,True,0.828
2,3,Avalon — Taku Iwasaki,base,1,0.000,JoJo,25,True,0.412
3,4,XL-TT — Hiroyuki Sawano,base,1,0.000,,,False,0.510
4,5,Jonathan Joestar theme — Hayato Matsuo,base,1,0.000,,,False,0.683
5,6,Time of the Decisive Battle — Yugo Kanno,base,1,0.000,,,True,0.788
6,7,Tense — Taku Iwasaki,base,1,0.000,JoJo,43,True,0.549
7,8,Il mare eterno nella mia anima ~Lunetta~ — Cao...,base,1,0.000,JoJo,9,True,0.109
8,9,Joestar Boys — Hayato Matsuo,base,1,0.000,JoJo,53,True,0.292
9,10,Sorrow — Yugo Kanno,base,1,0.000,JoJo,147,True,0.704


## Saved queue vs current-policy replay vs proposal

In [9]:
if "captured_labels" in result:
    horizon = max(len(result["captured_labels"]["saved"]), len(result["captured_labels"]["proposed"]))
    comparison = pd.DataFrame({
        "#": range(1, horizon + 1),
        "saved on phone": result["captured_labels"]["saved"],
        "proposed": result["captured_labels"]["proposed"],
    })
else:
    snapshot = result["data"]["snapshot"]
    def track_label(track_id):
        row = snapshot.row_by_id.get(track_id)
        if row is None:
            return f"[unresolved] {track_id}"
        meta = snapshot.meta[row]
        return f"{meta.get('title') or track_id} — {meta.get('artist') or 'Unknown artist'}"

    horizon = max(len(result["saved_future"]), len(result["current_ids"]), len(result["proposal_ids"]))
    comparison = pd.DataFrame({
        "#": range(1, horizon + 1),
        "saved on phone": [track_label(x) for x in result["saved_future"]] + [""] * (horizon - len(result["saved_future"])),
        "current hard replay": [track_label(x) for x in result["current_ids"]] + [""] * (horizon - len(result["current_ids"])),
        "proposed": [track_label(x) for x in result["proposal_ids"]] + [""] * (horizon - len(result["proposal_ids"])),
    })
display(comparison)


,#,saved on phone,current hard replay,proposed
0,1,Persistence ~Innocent Scream~ — Hayato Matsuo,Persistence ~Innocent Scream~ — Hayato Matsuo,Persistence ~Innocent Scream~ — Hayato Matsuo
1,2,Guardians of the Galaxy - Main Theme — Tyler B...,Clash — Yugo Kanno,Clash — Yugo Kanno
2,3,Clash — Yugo Kanno,Avalon — Taku Iwasaki,Avalon — Taku Iwasaki
3,4,Avalon — Taku Iwasaki,XL-TT — Hiroyuki Sawano,XL-TT — Hiroyuki Sawano
4,5,Jonathan Joestar theme — Hayato Matsuo,Jonathan Joestar theme — Hayato Matsuo,Jonathan Joestar theme — Hayato Matsuo
5,6,Il mare eterno nella mia anima ~Lunetta~ — Cao...,Time of the Decisive Battle — Yugo Kanno,Time of the Decisive Battle — Yugo Kanno
6,7,Sorrow — Yugo Kanno,Tense — Taku Iwasaki,Tense — Taku Iwasaki
7,8,Tense — Taku Iwasaki,Il mare eterno nella mia anima ~Lunetta~ — Cao...,Il mare eterno nella mia anima ~Lunetta~ — Cao...
8,9,Joestar Boys — Hayato Matsuo,Joestar Boys — Hayato Matsuo,Joestar Boys — Hayato Matsuo
9,10,Spider-Man: Homecoming Suite — Michael Giacchino,Sorrow — Yugo Kanno,Sorrow — Yugo Kanno


## Mechanical quality proxies and safety gates

In [10]:
metrics = result["metrics"].copy()
metrics["playlist share"] = metrics["playlist share"].map(lambda x: f"{x:.1%}")
for column in ["p10 transition cosine", "median transition cosine", "worst normalized regret"]:
    metrics[column] = metrics[column].map(lambda x: f"{x:.3f}")
for column in ["unique artists", "playlist promotions", "quota overrides", "injected selections"]:
    metrics[column] = metrics[column].astype(int)
display(metrics)

promotions = [row for row in result["proposal_diag"] if row["reason"] == "playlist_promotion"]
promotion_hops = [row["hop"] for row in promotions]
safety = pd.Series({
    "zero out-of-pool selections": True,
    "all promotions base rank <= 5": all(row["base_rank"] <= 5 for row in promotions),
    "all promotions normalized regret <= 0.20": all(row["normalized_regret"] <= 0.20 + 1e-9 for row in promotions),
    "no consecutive promotions": all(b - a > 1 for a, b in zip(promotion_hops, promotion_hops[1:])),
    "at most 2 promotions in any 12 hops": all(sum(start <= hop < start + 12 for hop in promotion_hops) <= 2 for start in range(1, 24)),
    "proposal equals playlist-off control for this seed": result["proposal_ids"] == result["off_ids"],
}, name="passed")
display(safety.to_frame())

,playlist share,p10 transition cosine,median transition cosine,unique artists,playlist promotions,quota overrides,injected selections,worst normalized regret
saved queue,60.9%,0.222,0.757,13,0,0,0,0.000
current hard replay,73.9%,0.117,0.446,12,0,1,1,1.546
playlist off,43.5%,0.349,0.788,16,0,0,0,0.000
proposed soft,43.5%,0.349,0.788,16,0,0,0,0.000


,passed
zero out-of-pool selections,True
all promotions base rank <= 5,True
all promotions normalized regret <= 0.20,True
no consecutive promotions,True
at most 2 promotions in any 12 hops,True
proposal equals playlist-off control for this seed,True


## Findings

In [11]:
hard = result["metrics"].loc["current hard replay"]
soft = result["metrics"].loc["proposed soft"]
n_promotions = sum(row["reason"] == "playlist_promotion" for row in result["proposal_diag"])
print(f"1. The proposed prior made {n_promotions} direct playlist promotions. For this seed it abstained: no playlist candidate was close enough to justify displacing the base SMART winner.")
print(f"2. Opted-playlist exposure fell from {hard['playlist share']:.1%} in the current-hard sensitivity replay to {soft['playlist share']:.1%}, while the playlist-off control was {result['metrics'].loc['playlist off', 'playlist share']:.1%}.")
print(f"3. The weakest-decile transition cosine rose from {hard['p10 transition cosine']:.3f} to {soft['p10 transition cosine']:.3f}; median transition cosine rose from {hard['median transition cosine']:.3f} to {soft['median transition cosine']:.3f}.")
print(f"4. Current-state replay matched only {result['overview']['saved/current exact position matches']} saved positions, so this is an approximate policy-sensitivity result, not an exact reconstruction or causal preference estimate.")
print("5. Recommendation: use this slate for listening review. If it sounds better, implement the relevance gate plus full-lookahead planning, then add decision provenance before tuning strength from skip/completion outcomes.")

1. The proposed prior made 0 direct playlist promotions. For this seed it abstained: no playlist candidate was close enough to justify displacing the base SMART winner.
2. Opted-playlist exposure fell from 73.9% in the current-hard sensitivity replay to 43.5%, while the playlist-off control was 43.5%.
3. The weakest-decile transition cosine rose from 0.117 to 0.349; median transition cosine rose from 0.446 to 0.788.
4. Current-state replay matched only 3/23 saved positions, so this is an approximate policy-sensitivity result, not an exact reconstruction or causal preference estimate.
5. Recommendation: use this slate for listening review. If it sounds better, implement the relevance gate plus full-lookahead planning, then add decision provenance before tuning strength from skip/completion outcomes.


## Limitations

- The existing history log has outcomes but no recommendation request ID, candidate ranks, quota/injection reason, or playlist attribution. It cannot tell whether a playlist intervention caused a skip.
- The saved queue lacks its generation timestamp, model/index/library hashes, snapshot row order, and chooser cache. The current-hard replay is therefore a sensitivity comparison using today's live inputs.
- Audio transition cosine, diversity, and score regret are mechanical proxies—not proof of listener satisfaction. The decisive next step is a listening review of the proposed slate, followed by locally logged, randomized switchback evaluation if the policy is implemented.